In [22]:
import numpy as np
import pandas as pd
import os
import csv
import math
import matplotlib.pyplot as plt
import datetime as dt
from datetime import datetime, timedelta
%matplotlib inline
import plotly as py
import plotly.figure_factory as ff

In [91]:
def clean_cp(cp_path,offset=0):
    # read the top of the CP CSV file 
    cp = pd.read_csv(cp_path,sep=',', skiprows=5, header=1)
    
    # extract CP start time 
    start_dt = cp.at[0, 'StartTime']
    start_dt = dt.datetime.strptime(start_dt, "%m/%d/%y %H:%M") + pd.to_timedelta(offset,unit='s')
    
    # read the top of the bottom of the CP CSV file 
    cp = pd.read_csv(cp_path,sep=',', skiprows=9, header=1)
    cp.drop(['Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10'], axis=1, inplace=True)
    cp['IPI + Duration'] = cp.apply(lambda col: col['IPI(ms)'] + col['Duration(ms)'], axis=1)
    cp['endtime rel'] = cp['IPI + Duration'].cumsum()
    cp['starttime rel'] = cp['endtime rel'] - cp['Duration(ms)']

    cp['starttime abs'] = cp['starttime rel'].apply(lambda x: start_dt + dt.timedelta(milliseconds=x))
    cp['endtime abs'] = cp['endtime rel'].apply(lambda x: start_dt + dt.timedelta(milliseconds=x))
    
    return cp

In [92]:
main_dir = "../Data/Pilot/"
all_p =["P"+str(x) for x in range(0,8)]
all_p =['P5']
# all_p.remove("P3")

for p in all_p:
    print(p)

    # Getting cp data
    cp_raw_path = os.path.join(main_dir,"Raw",p,"CP_1.csv")
    cp = clean_cp(cp_raw_path)
    # saving clean cp data to clean folder
    cp_clean_path = os.path.join(main_dir,'Clean',p,"CP.csv")
#     cp.to_csv(cp_clean_path)

P5


In [100]:
p='P5'
parts =[]
for i in ["1","2"]:
    print(p,i)

    # Getting cp data
    cp_raw_path = os.path.join(main_dir,"Raw",p,"CP_"+i+".csv")
    if i == "2":
        cp = clean_cp(cp_raw_path,25)
    else:
        cp = clean_cp(cp_raw_path)
    # saving clean cp data to clean folder
    
    parts += [cp] 
    
cp_clean_path = os.path.join(main_dir,'Clean',p,"CP.csv")
cp = pd.concat(parts)
cp.to_csv(cp_clean_path)

P5 1
P5 2


In [98]:
parts[1]["starttime abs"][0]

Timestamp('2020-03-02 18:06:04.606000')